## Этап 1. Input Data (Входные данные и первичный QA)

### 1.1. Загрузка и структура

Цель этапа:
1. Прочитать исходный датасет `sap500.csv`.
2. Проверить структуру и типы столбцов.
3. Привести поле `Date` к временному типу.
4. Отсортировать данные по дате по возрастанию.
5. Проверить монотонность, уникальность дат и наличие дубликатов.
6. Сохранить промежуточный артефакт для следующих этапов.

Важно: на этом этапе не выполняется содержательная обработка аномалий, дубликатов или пропусков. Эти задачи относятся к этапу 1.2.

In [1]:
import os
from pathlib import Path

import pandas as pd

# ============================================================
# 1. Константы этапа
# ============================================================
# Используем только реально существующие поля датасета.
# Порядок колонок фиксируем явно, чтобы избежать случайного
# использования лишних или отсутствующих столбцов.
REQUIRED_COLUMNS = [
    "Date",
    "Open",
    "High",
    "Low",
    "Close",
    "Volume",
]

RAW_FILE_NAME = "sap500.csv"


# ============================================================
# 2. Детерминированный поиск исходного файла
# ============================================================
# Исправление проблемы воспроизводимости:
# - не используем glob.glob('**/sap500.csv') как основной способ;
# - сначала проверяем явные, ожидаемые пути проекта;
# - если найдено несколько кандидатов, печатаем их и берём первый
#   согласно детерминированному порядку.
#
# Если в проекте используется нестандартное расположение файла,
# можно задать переменную окружения:
# SAP500_RAW_PATH=/полный/путь/к/sap500.csv
def resolve_raw_path(file_name: str) -> Path:
    """
    Ищет исходный файл детерминированно:
    1. Через переменную окружения SAP500_RAW_PATH, если она задана.
    2. В текущей рабочей директории.
    3. В стандартной структуре проекта: data/raw/.
    4. Вверх по дереву проекта не более чем на несколько уровней.
    """
    # Приоритет 0: явное задание пути через переменную окружения.
    env_path = os.getenv("SAP500_RAW_PATH")
    if env_path:
        p = Path(env_path).expanduser().resolve()
        if p.exists() and p.is_file():
            return p
        raise FileNotFoundError(
            f"Переменная окружения SAP500_RAW_PATH указывает на несуществующий файл: {p}"
        )

    cwd = Path.cwd().resolve()

    # Собираем кандидатов детерминированно.
    candidates = []

    # Текущая директория и несколько родительских уровней вверх.
    # Это покрывает случаи:
    # - запуск из корня проекта;
    # - запуск из папки notebooks/;
    # - запуск из папки, где лежит сам файл.
    bases = [cwd, *list(cwd.parents)[:4]]

    for base in bases:
        candidates.extend(
            [
                # Файл лежит прямо в текущей/родительской папке
                base / file_name,
                # Стандартная структура проекта: data/raw/
                base / "data" / "raw" / file_name,
                # Если файл случайно лежит внутри ноутбуковой папки
                base / "notebooks" / file_name,
            ]
        )

    # Убираем дубликаты путей, сохраняя порядок.
    seen = set()
    unique_candidates = []
    for candidate in candidates:
        resolved = candidate.resolve()
        if resolved not in seen:
            seen.add(resolved)
            unique_candidates.append(resolved)

    # Оставляем только реально существующие файлы.
    existing = [path for path in unique_candidates if path.is_file()]

    if not existing:
        raise FileNotFoundError(
            f"Файл {file_name} не найден.\n"
            f"Текущая рабочая директория: {cwd}\n"
            f"Проверенные пути:\n"
            + "\n".join(str(path) for path in unique_candidates)
        )

    if len(existing) > 1:
        print("Найдено несколько кандидатов на исходный файл:")
        for path in existing:
            print(" -", path)
        print("Будет использован первый кандидат согласно детерминированному порядку.")

    return existing[0]


RAW_PATH = resolve_raw_path(RAW_FILE_NAME)
print("Источник данных:", RAW_PATH)


# ============================================================
# 3. Чтение исходного датасета
# ============================================================
# Исходный файл не модифицируем.
# Все преобразования выполняются на копии в памяти.
df = pd.read_csv(RAW_PATH)

# Защита от случайных пробелов в именах колонок.
# Например: " Date" или "Close ".
df.columns = [str(col).strip() for col in df.columns]

# Проверяем, что все обязательные поля присутствуют.
missing_columns = [col for col in REQUIRED_COLUMNS if col not in df.columns]
if missing_columns:
    raise KeyError(
        f"В датасете отсутствуют обязательные столбцы: {missing_columns}"
    )

# Если есть лишние столбцы, фиксируем их и оставляем только нужные.
unexpected_columns = [col for col in df.columns if col not in REQUIRED_COLUMNS]
if unexpected_columns:
    print("Внимание: обнаружены лишние столбцы, они будут исключены:")
    print(unexpected_columns)

# Явно фиксируем порядок и состав столбцов.
df = df[REQUIRED_COLUMNS].copy()

print("Форма датасета:", df.shape)

print("Типы данных до преобразования:")
print(df.dtypes)

print("Первые строки до преобразования:")
print(df.head())


# ============================================================
# 4. Приведение столбца Date к временному типу
# ============================================================
# Используем errors='coerce', чтобы любые некорректные даты
# стали NaT и были явно видны в QA-проверке.
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# Приводим явно к datetime64[ns], чтобы избежать разночтений
# между разными версиями pandas.
df["Date"] = df["Date"].astype("datetime64[ns]")

nat_count = int(df["Date"].isna().sum())
print("Количество нераспознанных дат (NaT):", nat_count)

if nat_count > 0:
    print(
        "Внимание: обнаружены нераспознанные даты. "
        "Их обработка должна быть выполнена на этапе 1.2."
    )


# ============================================================
# 5. Строгая сортировка по Date по возрастанию
# ============================================================
# Критично для временных рядов:
# нарушение хронологии приводит к data leakage и некорректному
# расчёту лагов, доходностей и скользящих окон.
df = df.sort_values(
    by="Date",
    ascending=True,
    kind="mergesort",
).reset_index(drop=True)


# ============================================================
# 6. Проверки целостности временного ряда после сортировки
# ============================================================
print("Типы данных после преобразования:")
print(df.dtypes)

print("Минимальная дата:", df["Date"].min())
print("Максимальная дата:", df["Date"].max())

print("Date уникален:", df["Date"].is_unique)
print("Date монотонно возрастает:", df["Date"].is_monotonic_increasing)

print("Полные дубликаты строк:", int(df.duplicated().sum()))
print("Дубликаты по Date:", int(df["Date"].duplicated().sum()))

print("Первые строки после сортировки:")
print(df.head())

print("Последние строки после сортировки:")
print(df.tail())


# ============================================================
# 7. Технические assertions для этапа 1.1
# ============================================================
# Эти проверки защищают от скрытых проблем при перезапуске ноутбука.
assert df.shape[0] > 0, "Датасет пуст: после чтения нет строк."
assert df.shape[1] == len(REQUIRED_COLUMNS), "Нарушена структура столбцов."

# Если есть NaT, монотонность может быть некорректной.
# В этом случае проблема должна разбираться на этапе 1.2.
if nat_count == 0:
    assert df["Date"].is_monotonic_increasing, (
        "После сортировки даты не монотонно возрастают."
    )


# ============================================================
# 8. Сохранение промежуточного артефакта
# ============================================================
# Исправление риска воспроизводимости:
# результат этапа 1.1 сохраняем в data/processed/,
# чтобы следующие этапы могли читать уже отсортированные данные.
def resolve_processed_dir(raw_path: Path) -> Path:
    """
    Определяет папку для обработанных данных.

    Если исходный файл лежит в стандартной структуре:
        .../data/raw/sap500.csv
    то обработанные данные кладём рядом:
        .../data/processed/

    Иначе создаём data/processed относительно текущей рабочей директории.
    """
    raw_path = raw_path.resolve()

    if (
        raw_path.parent.name == "raw"
        and raw_path.parent.parent.name == "data"
    ):
        return raw_path.parent.parent / "processed"

    return Path.cwd().resolve() / "data" / "processed"


PROCESSED_DIR = resolve_processed_dir(RAW_PATH)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

PROCESSED_PARQUET_PATH = PROCESSED_DIR / "sap500_stage1_1_sorted.parquet"

# Parquet предпочтителен, так как сохраняет типы данных.
# Если в окружении нет движка Parquet, сохраняем fallback в CSV.
try:
    df.to_parquet(PROCESSED_PARQUET_PATH, index=False)
    print("Артефакт сохранён:", PROCESSED_PARQUET_PATH)
except Exception as exc:
    PROCESSED_CSV_PATH = PROCESSED_DIR / "sap500_stage1_1_sorted.csv"
    df.to_csv(PROCESSED_CSV_PATH, index=False)
    print("Parquet-сохранение не удалось. Артефакт сохранён как CSV:", PROCESSED_CSV_PATH)
    print("Причина ошибки Parquet:", exc)

Источник данных: /Users/ernstavraev/Downloads/датасеты/sp500_volatility_project/data/raw/sap500.csv
Форма датасета: (24809, 6)
Типы данных до преобразования:
Date       object
Open      float64
High      float64
Low       float64
Close     float64
Volume    float64
dtype: object
Первые строки до преобразования:
         Date       Open       High        Low      Close  Volume
0  1927-12-30  17.660000  17.660000  17.660000  17.660000     0.0
1  1928-01-03  17.760000  17.760000  17.760000  17.760000     0.0
2  1928-01-04  17.719999  17.719999  17.719999  17.719999     0.0
3  1928-01-05  17.549999  17.549999  17.549999  17.549999     0.0
4  1928-01-06  17.660000  17.660000  17.660000  17.660000     0.0
Количество нераспознанных дат (NaT): 0
Типы данных после преобразования:
Date      datetime64[ns]
Open             float64
High             float64
Low              float64
Close            float64
Volume           float64
dtype: object
Минимальная дата: 1927-12-30 00:00:00
Максимальная дат

## Промежуточный вывод по Этапу 1.1

### Что сделано на этапе

1. Исходный датасет `sap500.csv` прочитан без модификации исходного файла.
2. Имена столбцов очищены от случайных пробелов.
3. Проверено наличие обязательных полей:
   - `Date`;
   - `Open`;
   - `High`;
   - `Low`;
   - `Close`;
   - `Volume`.
4. Поле `Date` приведено к типу `datetime64[ns]`.
5. Данные отсортированы по дате по возрастанию.
6. Индекс сброшен, чтобы избежать скрытых эффектов от старой индексации.
7. Проверены:
   - диапазон дат;
   - уникальность дат;
   - монотонность дат;
   - полные дубликаты строк;
   - дубликаты по `Date`.
8. Результат этапа сохранён в `data/processed/`.